# This notebook will transform:

Operations
↓
Financial Results

# by generating:

Fuel Costs
Crew Costs
Airport Fees
Maintenance Costs
Navigation Costs

↓

Total Operating Cost

↓

Operating Profit

↓

Operating Margin %

In [63]:
import pandas as pd
import numpy as np

OUTPUT_FOLDER = "../Data/Generated/"

In [64]:
fact = pd.read_csv(
    OUTPUT_FOLDER + "fact_operations_stage1.csv"
)

print(fact.shape)

(56992, 43)


In [65]:
fact.head()

,Date,Year,Quarter,Month,Month_Name,Week_Number,Day_Of_Week,Is_Weekend,Season,Route_ID,...,Route_Demand_Multiplier,Expected_Passengers,Passengers,Load_Factor,Fare_Position,Avg_Fare_USD,Ticket_Revenue_USD,Ancillary_Per_Pax,Ancillary_Revenue_USD,Total_Revenue_USD
0,2020-01-01,2020,1,1,January,1,Wednesday,False,Winter,1,...,3.3,580.547625,563.0,0.853030,0.911818,207.407407,116770.370324,29.651106,16693.572661,133463.942985
1,2020-01-01,2020,1,1,January,1,Wednesday,False,Winter,2,...,3.3,490.830133,444.0,0.672727,0.803636,206.362131,91624.786187,22.975415,10201.084186,101825.870373
2,2020-01-01,2020,1,1,January,1,Wednesday,False,Winter,3,...,2.8,359.889049,346.0,0.915344,0.949206,200.621489,69415.035349,24.395891,8440.978202,77856.013552
3,2020-01-01,2020,1,1,January,1,Wednesday,False,Winter,4,...,2.2,162.397904,152.0,0.804233,0.882540,185.812810,28243.547162,27.628012,4199.457786,32443.004948
4,2020-01-01,2020,1,1,January,1,Wednesday,False,Winter,5,...,2.2,121.755700,126.0,0.666667,0.800000,174.069588,21932.768072,27.961988,3523.210482,25455.978554


In [66]:
fact.columns.to_list()


['Date',
 'Year',
 'Quarter',
 'Month',
 'Month_Name',
 'Week_Number',
 'Day_Of_Week',
 'Is_Weekend',
 'Season',
 'Route_ID',
 'City',
 'Country',
 'Region',
 'Distance_KM',
 'Route_Tier',
 'Seasonal_Route',
 'Launch_Date',
 'Base_Daily_Demand',
 'Fare_Low_USD',
 'Fare_High_USD',
 'Aircraft_Type',
 'Base_Daily_Flights',
 'Competition_Sensitivity',
 'Fuel_Index',
 'Security_Risk_Index',
 'Holiday_Name',
 'Demand_Impact',
 'Seats',
 'Fuel_Burn_Index',
 'Demand_Index',
 'Competition_Index',
 'Flights_Operated',
 'Seats_Offered',
 'Route_Demand_Multiplier',
 'Expected_Passengers',
 'Passengers',
 'Load_Factor',
 'Fare_Position',
 'Avg_Fare_USD',
 'Ticket_Revenue_USD',
 'Ancillary_Per_Pax',
 'Ancillary_Revenue_USD',
 'Total_Revenue_USD']

In [67]:
aircraft_cost_factor = {
    "B737-800": 1.00,
    "B737-900ER": 1.08,
    "B737-MAX8": 0.92
}

fact["Aircraft_Cost_Factor"] = (
    fact["Aircraft_Type"]
    .map(aircraft_cost_factor)
)

In [68]:
# Fuel cost calculation:
fact["Fuel_Cost_USD"] = (
    fact["Distance_KM"]
    *
    fact["Flights_Operated"]
    *
    fact["Fuel_Burn_Index"]
    *
    fact["Fuel_Index"]
    *
    0.07
)

In [69]:
# Crew cost calculation:
fact["Crew_Cost_USD"] = (
    fact["Flights_Operated"]
    *
    (
        1000
        +
        (
            fact["Distance_KM"]
            * 0.10
        )
    )
    *
    fact["Aircraft_Cost_Factor"]
)

In [70]:
# Aircraft fees calculation:
fact["Airport_Fees_USD"] = (
    (
        fact["Passengers"]
        * 16
    )
    +
    (
        fact["Flights_Operated"]
        * 900
    )
)

In [71]:
# Maintenance cost calculation:
fact["Maintenance_Cost_USD"] = (
    fact["Flights_Operated"]
    *
    fact["Distance_KM"]
    *
    0.55
    *
    fact["Aircraft_Cost_Factor"]
)

In [72]:
# Navigation fees calculation:
fact["Navigation_Fees_USD"] = (
    fact["Flights_Operated"]
    *
    fact["Distance_KM"]
    *
    0.25
)

In [73]:
# Ground Handling cost calculation:
fact["Ground_Handling_Cost_USD"] = (
    fact["Passengers"]
    * 12
)

In [74]:
# Aircraft Lease cost calculation:
fact["Aircraft_Lease_Cost_USD"] = (
    fact["Flights_Operated"]
    * 3500
)

In [75]:
# Security cost calculation:
fact["Security_Cost_USD"] = (
    fact["Passengers"]
    *
    (
        4
        +
        fact["Security_Risk_Index"]
    )
)

In [76]:
# Corporate overhead calculation:
fact["Corporate_Overhead_USD"] = (
    fact["Total_Revenue_USD"] * 0.08
)

In [77]:
# Calculationg Total cost:
fact["Total_Operating_Cost_USD"] = (
    fact["Fuel_Cost_USD"]
    + fact["Crew_Cost_USD"]
    + fact["Airport_Fees_USD"]
    + fact["Maintenance_Cost_USD"]
    + fact["Navigation_Fees_USD"]
    + fact["Ground_Handling_Cost_USD"]
    + fact["Aircraft_Lease_Cost_USD"]
    + fact["Security_Cost_USD"]
    + fact["Corporate_Overhead_USD"]
)

In [78]:
# Profit:
fact["Operating_Profit_USD"] = (
    fact["Total_Revenue_USD"]
    -
    fact["Total_Operating_Cost_USD"]
)

In [79]:
# Profit Margin:
fact["Operating_Margin_Pct"] = (
    fact["Operating_Profit_USD"]
    /
    fact["Total_Revenue_USD"]
)

fact["Operating_Margin_Pct"] *= 100

In [80]:
# Validatoion of results:
print(
    "Revenue:",
    round(
        fact["Total_Revenue_USD"].sum()/1_000_000,
        2
    ),
    "Million USD"
)

print(
    "Cost:",
    round(
        fact["Total_Operating_Cost_USD"].sum()/1_000_000,
        2
    ),
    "Million USD"
)

print(
    "Profit:",
    round(
        fact["Operating_Profit_USD"].sum()/1_000_000,
        2
    ),
    "Million USD"
)

print(
    "Average Margin:",
    round(
        fact["Operating_Margin_Pct"].mean(),
        2
    ),
    "%"
)

Revenue: 4138.42 Million USD
Cost: 3031.45 Million USD
Profit: 1106.97 Million USD
Average Margin: 22.53 %


In [81]:
cost_breakdown = fact[
[
    "Fuel_Cost_USD",
    "Crew_Cost_USD",
    "Airport_Fees_USD",
    "Maintenance_Cost_USD",
    "Navigation_Fees_USD",
    "Ground_Handling_Cost_USD",
    "Aircraft_Lease_Cost_USD",
    "Security_Cost_USD"
]
].sum() / 1_000_000

cost_breakdown.sort_values(
    ascending=False
)

Fuel_Cost_USD               1349.703910
Aircraft_Lease_Cost_USD      383.712000
Airport_Fees_USD             367.597760
Ground_Handling_Cost_USD     201.696720
Security_Cost_USD            130.609188
Crew_Cost_USD                124.884272
Maintenance_Cost_USD          96.432778
Navigation_Fees_USD           45.739420
dtype: float64

In [82]:
fact[
[
    "Fuel_Cost_USD",
    "Crew_Cost_USD",
    "Airport_Fees_USD",
    "Maintenance_Cost_USD",
    "Navigation_Fees_USD"
]
].describe()

,Fuel_Cost_USD,Crew_Cost_USD,Airport_Fees_USD,Maintenance_Cost_USD,Navigation_Fees_USD
count,56992.000000,56992.000000,56992.000000,56992.000000,56992.000000
mean,23682.339804,2191.259691,6449.988770,1692.040598,802.558605
std,13034.555622,921.727908,3075.916441,869.413730,433.636813
min,4428.942000,1105.000000,2084.000000,561.000000,255.000000
25%,13927.554900,1185.420000,3764.000000,1087.900000,537.500000
50%,20700.611820,2270.000000,5928.000000,1459.810000,705.000000
75%,30952.048050,2573.240000,7848.000000,2138.400000,1042.500000
max,74521.604080,4136.000000,15696.000000,4032.820000,1992.500000


KPIs Already Available

You can now calculate:

Revenue
Cost
Profit
Margin
Load Factor
Yield
Revenue per Passenger
Revenue per Route
Revenue by Country
Revenue by Region
Revenue by Season
Cost per Passenger
Cost per Flight
Profitability by Route

without generating any additional data.

In [83]:
fact.shape

(56992, 56)

In [84]:
fact.to_csv(
    OUTPUT_FOLDER +
    "fact_financials.csv",
    index=False
)

print("fact_financials.csv saved")

fact_financials.csv saved
